# 32. Veto-aware densities and signal PDFs

**Objectives:**
- Apply a veto map to an arbitrary Dalitz shape with `VetoedDensity`.
- Build a veto-aware `SignalPDF` directly from a `DecayModel` with `vetoed_signal_pdf`.
- Compare the veto-aware normalized density against the unvetoed one, inside and
  outside the vetoed region.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. This continues from
[tutorial 31](tutorial_31_vetoes.ipynb); see `docs/backgrounds_and_vetoes.md`.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np
import jax.numpy as jnp
from dalitzplotfitter import (
    DecayChannel, DecayModel, MassWindowVeto, NonResonant, RealImag,
    Resonance, VetoedDensity, generate_toy, vetoed_signal_pdf,
)

## 1. A small model and a veto

Same `B+ -> K+ pi+ pi-` model as tutorial 31, with a `MassWindowVeto` on the
`K+ pi-` (`pair=(0, 2)`) system between 1.40 and 1.60 GeV.

In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
model = DecayModel(
    channel,
    [
        Resonance("Kstar", (0, 2), RealImag(1.0, 0.0), mass=0.892, width=0.051, spin=1),
        NonResonant(RealImag(0.4, -0.2), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 2),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in model.parameters}
veto = MassWindowVeto((0, 2), 1.40, 1.60)
print(f"Normalization grid has {model.normalization_sample.size} points.")

Normalization grid has 3600 points.


## 2. `VetoedDensity`: veto an arbitrary shape

`VetoedDensity` multiplies any callable density by the veto's acceptance
mask; it is the building block used for backgrounds in
`docs/backgrounds_and_vetoes.md` ("For a background shape, wrap it with the
same veto"). Here we veto a simple smooth combinatorial-background shape and
confirm it is exactly zero inside the vetoed `K+ pi-` mass window and
unchanged outside it.

In [3]:
def background_shape(data):
    return 1.0 + 0.3 * data["s12"] / channel.parent_mass**2

vetoed_background = VetoedDensity(background_shape, veto)

grid = model.normalization_sample.as_dict()
mask = np.asarray(veto.accept(grid))
plain_values = np.asarray(background_shape(grid))
vetoed_values = np.asarray(vetoed_background(grid))

np.testing.assert_allclose(vetoed_values[mask], plain_values[mask])
np.testing.assert_array_equal(vetoed_values[~mask], 0.0)
print(f"Vetoed grid points: {int((~mask).sum())} / {mask.size}; "
      f"all set to zero: {bool(np.all(vetoed_values[~mask] == 0.0))}")

Vetoed grid points: 60 / 3600; all set to zero: True


## 3. `vetoed_signal_pdf`: a veto-aware `SignalPDF` from a `DecayModel`

`vetoed_signal_pdf(model, veto)` builds a `SignalPDF` whose intensity is
`model.intensity`, integrated with a `GridIntegrator` on
`model.normalization_sample` by default, with the veto entering both the
numerator (rejected events get zero density) and the normalization integral
(so the density still integrates to one over the accepted region).

In [4]:
pdf = vetoed_signal_pdf(model, veto)

density = np.asarray(pdf(grid, truth))
print(f"Density is exactly zero on all {int((~mask).sum())} vetoed grid points: "
      f"{bool(np.all(density[~mask] == 0.0))}")
print(f"Density is strictly positive on all accepted grid points: "
      f"{bool(np.all(density[mask] > 0.0))}")

integral_over_accepted = float(np.mean(np.asarray(model.normalization_sample.weights)[mask] * density[mask]))
print(f"mean(weight * density) over the accepted region: {integral_over_accepted:.6f} (expect ~1)")

Density is exactly zero on all 60 vetoed grid points: True
Density is strictly positive on all accepted grid points: True
mean(weight * density) over the accepted region: 1.016949 (expect ~1)


## 4. Density with vs. without the veto

Without a veto, `model.intensity` is nonzero everywhere on the Dalitz plot,
including inside the vetoed window; the veto-aware PDF from step 3 is exactly
zero there instead. The two normalized densities also differ slightly outside
the vetoed region, since removing probability mass from the vetoed window
raises the normalization-weighted density everywhere else.

In [5]:
from dalitzplotfitter import SignalPDF
from dalitzplotfitter.integration import GridIntegrator

unvetoed_pdf = SignalPDF(
    intensity=lambda data, parameters: model.intensity(data, parameters),
    integrator=GridIntegrator(model.normalization_sample),
)
unvetoed_density = np.asarray(unvetoed_pdf(grid, truth))

print(f"Unvetoed density is nonzero inside the vetoed window: "
      f"{bool(np.all(unvetoed_density[~mask] > 0.0))}")
print("Mean density outside the vetoed window, unvetoed PDF: "
      f"{float(np.mean(unvetoed_density[mask])):.6f}")
print("Mean density outside the vetoed window, veto-aware PDF: "
      f"{float(np.mean(density[mask])):.6f}")

Unvetoed density is nonzero inside the vetoed window: True
Mean density outside the vetoed window, unvetoed PDF: 0.013137
Mean density outside the vetoed window, veto-aware PDF: 0.013321


## Try it yourself

1. Replace `MassWindowVeto` with a `CompositeVeto` and rebuild `vetoed_signal_pdf`.
2. Pass `normalization_sample=` explicitly to `vetoed_signal_pdf` with a finer grid.
3. Wrap a `Resonance`-only shape (not the full model) with `VetoedDensity` and plot it.

## Continue learning

Next: [Square-Dalitz SCF migration maps](tutorial_33_scf_map_dense.ipynb).
Reference: [backgrounds and vetoes](../../docs/backgrounds_and_vetoes.md).

Return to [the course guide](TUTORIALS.md).